In [1]:
!pip install fastapi uvicorn transformers accelerate sentencepiece bitsandbytes


In [1]:
import requests
from dotenv import load_dotenv
import os
import spacy
import xml.etree.ElementTree as ET
import json
import requests
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue
import os
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue
load_dotenv()

True

In [ ]:
import os
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from label_studio_sdk import LabelStudio

# =====================================================
# CONFIG
# =====================================================

LABEL_STUDIO_URL = "http://localhost:8080"
LABEL_STUDIO_API_KEY = os.getenv("LABEL_STUDIO")
PROJECT_ID = 2

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK")

MODEL_NAME = "deepseek-chat"

BATCH_SIZE = 4
MAX_WORKERS = 2

LABELS = [
    "INVENTION","COMPONENT","SUBSYSTEM","MATERIAL","CHEMICAL","BIOMOLECULE","COMPOSITION",
    "PROCESS_STEP","METHOD","PARAMETER","MEASUREMENT","CONDITION","FUNCTION","SIGNAL",
    "CONTROL","SOFTWARE","HARDWARE","FIGURE_REF","CLAIM_ELEMENT","PRIOR_ART","UNCLASSIFIED_ENTITY",
]

# =====================================================
# CLIENTS
# =====================================================

ls_client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=LABEL_STUDIO_API_KEY)

for p in ls_client.projects.list():
    print(p.id, p.title)

project = ls_client.projects.get(id=PROJECT_ID)
tasks = list(ls_client.tasks.list(project=project.id))
print("TASKS FOUND:", len(tasks))

create_lock = threading.Lock()

# One OpenAI client per thread (safe)
thread_local = threading.local()
def get_openai_client():
    if not hasattr(thread_local, "client"):
        thread_local.client = OpenAI(
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com",
        )
    return thread_local.client

# =====================================================
# UTILS
# =====================================================

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def build_prompt(text: str) -> str:
    labels_str = ", ".join(LABELS)
    return f"""
You are a high-recall patent NER engine.

Extract span entities from the input text.
Use ONLY these labels:
{labels_str}

Rules:
- Return ONLY a valid JSON array.
- Each item must have keys: "text" and "label".
- "text" must be copied verbatim from the input.
- High recall is preferred.
- Overlapping spans are allowed.

Input text:
\"\"\"{text}\"\"\"

Return JSON array now:
""".strip()

def entities_to_spans(text, entities):
    spans = []
    for e in entities:
        ent_text = e.get("text", "")
        label = e.get("label", "")
        if not ent_text or label not in LABELS:
            continue
        start = text.find(ent_text)
        if start == -1:
            continue
        spans.append({
            "start": start,
            "end": start + len(ent_text),
            "text": ent_text,
            "labels": [label],
        })
    return spans

# =====================================================
# DEEPSEEK CALL
# =====================================================

def call_deepseek(text: str):
    client = get_openai_client()

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You extract entities from patent text."},
            {"role": "user", "content": build_prompt(text)},
        ],
        temperature=0.0,
    )

    content = response.choices[0].message.content
    try:
        return json.loads(content)
    except Exception:
        print("JSON parse failed:", content)
        return []

# =====================================================
# BATCH PROCESSING
# =====================================================

def process_batch(task_batch):
    for task in task_batch:
        task_id = task.id
        text = (task.data or {}).get("text", "")
        if not text:
            continue

        print("CALLING API for task", task_id)

        try:
            entities = call_deepseek(text)
            spans = entities_to_spans(text, entities)

            with create_lock:
                ls_client.predictions.create(
                    task=task_id,
                    model_version="deepseek",
                    score=1.0,
                    result=[
                        {
                            "from_name": "label",
                            "to_name": "text",
                            "type": "labels",
                            "value": span,
                        }
                        for span in spans
                    ],
                )

            print("CREATED prediction for task", task_id)

        except Exception as e:
            print("ERROR task", task_id, "->", repr(e))

# =====================================================
# RUN
# =====================================================

batches = list(chunked(tasks, BATCH_SIZE))
print("BATCHES:", len(batches))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(process_batch, b) for b in batches]
    print("FUTURES SUBMITTED:", len(futures))

    for i, fut in enumerate(as_completed(futures), 1):
        fut.result()
        if i % 25 == 0:
            print("DONE", i, "/", len(futures))


3 THESIS Sentence Classifier
2 Thesis NER
TASKS FOUND: 13003
BATCHES: 3251
CALLING API for task 14021
CALLING API for task 14025
FUTURES SUBMITTED: 3251
ERROR task 14021 -> APIStatusError("Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}")
CALLING API for task 14022
ERROR task 14025 -> APIStatusError("Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}")
CALLING API for task 14026
ERROR taskERROR task 14022 -> APIStatusError("Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}")
CALLING API for task 14023
 14026 -> APIStatusError("Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}")
CALLING API for task 14027
ERROR taskERROR task 14023 -> APISt

In [2]:
##testing

import requests

MODEL_URL = "https://empiristic-mariyah-unprophetically.ngrok-free.dev/predict"

payload = {
    "data": [
        {"text": "A temperature sensor is connected to a control unit via a wireless interface."}
    ]
}

resp = requests.post(MODEL_URL, json=payload, timeout=60)
resp.raise_for_status()

print("Status:", resp.status_code)
print("Response JSON:")
print(resp.json())


Status: 200
Response JSON:
[{'result': [{'from_name': 'label', 'to_name': 'text', 'type': 'labels', 'value': {'start': 2, 'end': 20, 'text': 'temperature sensor', 'labels': ['COMPONENT']}}, {'from_name': 'label', 'to_name': 'text', 'type': 'labels', 'value': {'start': 39, 'end': 51, 'text': 'control unit', 'labels': ['COMPONENT']}}, {'from_name': 'label', 'to_name': 'text', 'type': 'labels', 'value': {'start': 58, 'end': 76, 'text': 'wireless interface', 'labels': ['SUBSYSTEM']}}], 'score': 1.0}]


In [ ]:

# ---------------- CONFIG ----------------
LABEL_STUDIO_URL = "http://localhost:8081"   # EXACT URL you open LS in browser
LABEL_STUDIO_API_KEY = os.getenv("LABEL_STUDIO")
PROJECT_ID = 2

MODEL_URL = "https://empiristic-mariyah-unprophetically.ngrok-free.dev/predict"
MODEL_VERSION = "mistral8B"

# Control names from your Label Studio labeling config
FROM_NAME = "ner"     # e.g. <Labels name="ner" ...>
TO_NAME = "text"      # e.g. <Text name="text" ...>

client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=LABEL_STUDIO_API_KEY.strip())
project = client.projects.get(id=PROJECT_ID)

# optional: keep this if you want parity with docs (not required anymore)
li = project.get_label_interface()

print(f"Connected. Project={project.id}")

for task in client.tasks.list(project=project.id):
    task_id = task.id
    data = task.data or {}

    text = data.get("text", "")
    if not isinstance(text, str) or not text.strip():
        print(f"Task {task_id}: no 'text' field, skipping.")
        continue

    # Call your Colab FastAPI /predict (expects {"data":[{"text":...}]})
    resp = requests.post(
        MODEL_URL,
        json={"data": [{"text": text}]},
        timeout=300,
    )

    # If something is wrong, this prints the FastAPI validation error
    if resp.status_code != 200:
        print(f"Task {task_id}: model error {resp.status_code}: {resp.text[:500]}")
        resp.raise_for_status()

    model_preds = resp.json()
    if isinstance(model_preds, dict):
        model_preds = [model_preds]

    if not model_preds:
        print(f"Task {task_id}: model returned empty list.")
        continue

    # Your /predict returns a list of prediction dicts like:
    # {"result": [...], "score": 1.0, "model_version": "..."}
    # We'll store each one as an LS Prediction.
    saved_any = False
    for mp in model_preds:
        result = mp.get("result")
        if not isinstance(result, list) or len(result) == 0:
            continue

        prediction = PredictionValue(
            model_version=mp.get("model_version", MODEL_VERSION),
            score=float(mp.get("score", 0.0)),
            result=result,  # <-- pass through as-is
        )

        client.predictions.create(task=task_id, **prediction.model_dump())
        saved_any = True

    if saved_any:
        print(f"✔ Saved prediction(s) for task {task_id}")
    else:
        # Debug: show the first model prediction keys to see what's missing
        print(f"Task {task_id}: no usable 'result'. First pred keys: {list(model_preds[0].keys())}")


Connected. Project=2
Task 14021: no usable 'result'. First pred keys: ['result', 'score', 'model_version']
Task 14022: no usable 'result'. First pred keys: ['result', 'score', 'model_version']
Task 14023: no usable 'result'. First pred keys: ['result', 'score', 'model_version']
Task 14024: no usable 'result'. First pred keys: ['result', 'score', 'model_version']
Task 14025: no usable 'result'. First pred keys: ['result', 'score', 'model_version']


KeyboardInterrupt: 